# Frost Expected Loss

**Goal:** Estimate E[Loss] = P(frost event) × E[loss | frost event] using ERA5 degree-hours and FAOSTAT production data.

**Phenological thresholds used (from existing trigger code):**
- Feb – early March: buds closed, threshold ~−3 °C
- Late March: −3 °C
- April: −1.5 °C (open catkins, high sensitivity)
- May: essentially zero tolerance

**Degree-hours (DH):** accumulated temperature deficit below threshold over the spring risk window. Higher DH → more frost damage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load Data

In [ ]:
frost = pd.read_csv("../data/raw/era5_frost_monthly.csv")
prod  = pd.read_csv("../data/raw/faostat/turkey_hazelnut_production.csv")

print(frost.shape, frost.columns.tolist())
print(prod.shape,  prod.columns.tolist())

In [ ]:
# Merge on overlapping years (production starts 1961)
df = frost.merge(prod, on="year", how="inner")
print(f"Panel: {df.year.min()}–{df.year.max()}, n={len(df)}")
df.head()

## 2. Frost Event Thresholds

Define three severity tiers based on domain knowledge of *Corylus avellana* cold tolerance.
These map directly to the payout bands used in the trigger contract.

In [ ]:
# DH thresholds (degree-hours) — calibrated to phenological windows in frost.py
TIER_MILD     = 50    # any measurable spring frost stress
TIER_MODERATE = 150   # material bud/catkin damage expected
TIER_SEVERE   = 300   # extensive damage; corresponds to events like 1985, 2007

df["frost_event"]    = df["frost_dh"] >= TIER_MILD
df["frost_moderate"] = df["frost_dh"] >= TIER_MODERATE
df["frost_severe"]   = df["frost_dh"] >= TIER_SEVERE

n = len(df)
for label, col, thresh in [("Mild (≥50 DH)", "frost_event", TIER_MILD),
                            ("Moderate (≥150 DH)", "frost_moderate", TIER_MODERATE),
                            ("Severe (≥300 DH)", "frost_severe", TIER_SEVERE)]:
    cnt = df[col].sum()
    print(f"{label:25s}: {cnt}/{n} years  →  P = {cnt/n:.3f}")

## 3. Production Loss Conditional on Frost

Use year-over-year production **change** to isolate weather signal from trend.
Positive `prod_chg` = production grew vs. prior year; negative = loss.

In [ ]:
df = df.sort_values("year").reset_index(drop=True)
df["prod_chg"] = df["production_mt"].pct_change()      # fractional YoY change
df["prod_chg_mt"] = df["production_mt"].diff()          # absolute MT change

# Drop the first row (NaN diff)
dfc = df.dropna(subset=["prod_chg"]).copy()

frost_yrs  = dfc[dfc["frost_event"]]
nofrost_yrs = dfc[~dfc["frost_event"]]

print(f"Mean YoY prod change — frost years:    {frost_yrs['prod_chg'].mean():+.2%}")
print(f"Mean YoY prod change — no-frost years: {nofrost_yrs['prod_chg'].mean():+.2%}")
print()
print(f"Median YoY prod change — frost years:    {frost_yrs['prod_chg'].median():+.2%}")
print(f"Median YoY prod change — no-frost years: {nofrost_yrs['prod_chg'].median():+.2%}")

In [ ]:
# t-test: are frost-year production changes significantly worse?
t, p = stats.ttest_ind(frost_yrs["prod_chg"], nofrost_yrs["prod_chg"], equal_var=False)
print(f"Welch t-test: t={t:.2f}, p={p:.3f}")

## 4. Degree-Hours vs. Production Change (scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(dfc["frost_dh"], dfc["prod_chg"] * 100, alpha=0.7, edgecolors="none", s=40, color="steelblue")

# OLS trend line
m, b, r, p_val, _ = stats.linregress(dfc["frost_dh"], dfc["prod_chg"] * 100)
xr = np.linspace(0, dfc["frost_dh"].max() * 1.05, 200)
ax.plot(xr, m * xr + b, "--", color="tomato", lw=1.5, label=f"OLS slope={m:.3f}%/DH, r²={r**2:.2f}, p={p_val:.3f}")

for thresh, lbl in [(TIER_MILD, "Mild"), (TIER_MODERATE, "Mod."), (TIER_SEVERE, "Severe")]:
    ax.axvline(thresh, color="gray", lw=0.8, ls=":")
    ax.text(thresh + 3, ax.get_ylim()[1] if ax.get_ylim()[1] else 60, lbl, fontsize=7, color="gray")

ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("Total Frost Degree-Hours (spring window)")
ax.set_ylabel("YoY Production Change (%)")
ax.set_title("Frost Stress vs. Hazelnut Production Change — Turkey")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. E[Loss | Frost] by Severity Tier

Conditional expected production loss as a fraction of prior-year production.

In [ ]:
tiers = [
    ("No frost (<50 DH)",    ~dfc["frost_event"]),
    ("Mild [50–150 DH)",     dfc["frost_event"] & ~dfc["frost_moderate"]),
    ("Moderate [150–300 DH)", dfc["frost_moderate"] & ~dfc["frost_severe"]),
    ("Severe (≥300 DH)",      dfc["frost_severe"]),
]

rows = []
for label, mask in tiers:
    sub = dfc[mask]
    n_sub = len(sub)
    mean_chg = sub["prod_chg"].mean()
    # loss = negative production change; clamp at 0 (we only care about downside)
    mean_loss = sub["prod_chg"].clip(upper=0).abs().mean()
    rows.append({"Tier": label, "N": n_sub, "Mean prod chg": mean_chg, "E[loss | tier]": mean_loss})

tier_df = pd.DataFrame(rows)
tier_df["Mean prod chg"] = tier_df["Mean prod chg"].map("{:+.2%}".format)
tier_df["E[loss | tier]"] = tier_df["E[loss | tier]"].map("{:.2%}".format)
print(tier_df.to_string(index=False))

## 6. Expected Loss = P(tier) × E[Loss | tier]

In [ ]:
n_total = len(dfc)

el_rows = []
for label, mask in tiers:
    sub = dfc[mask]
    p_tier = len(sub) / n_total
    e_loss_given_tier = sub["prod_chg"].clip(upper=0).abs().mean()
    el = p_tier * e_loss_given_tier
    el_rows.append({"Tier": label, "P(tier)": p_tier, "E[loss|tier]": e_loss_given_tier, "EL contribution": el})

el_df = pd.DataFrame(el_rows)
total_el = el_df["EL contribution"].sum()

print(el_df.assign(
    **{"P(tier)": el_df["P(tier)"].map("{:.3f}".format),
       "E[loss|tier]": el_df["E[loss|tier]"].map("{:.2%}".format),
       "EL contribution": el_df["EL contribution"].map("{:.3%}".format)}
).to_string(index=False))
print(f"\nTotal Annual Expected Loss (% of production): {total_el:.3%}")

## 7. EL Breakdown Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels = el_df["Tier"].str.replace(" (", "\n(", regex=False).str.replace(" [", "\n[", regex=False)
colors = ["#90caf9", "#ffcc80", "#ef9a9a", "#b71c1c"]

# P(tier) bars
axes[0].bar(labels, el_df["P(tier)"], color=colors, edgecolor="white")
axes[0].set_title("Historical Frequency by Tier")
axes[0].set_ylabel("Probability")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# EL contribution bars
axes[1].bar(labels, el_df["EL contribution"] * 100, color=colors, edgecolor="white")
axes[1].set_title("EL Contribution by Tier (% of production)")
axes[1].set_ylabel("Expected Loss (%)")

for ax in axes:
    ax.tick_params(axis="x", labelsize=8)

plt.suptitle(f"Frost Expected Loss — Total EL = {total_el:.2%} of annual production", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 8. DH Distribution — Kernel Density

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

from scipy.stats import gaussian_kde
dh_vals = df["frost_dh"].values
kde = gaussian_kde(dh_vals, bw_method=0.4)
xgrid = np.linspace(0, dh_vals.max() * 1.05, 400)
ax.fill_between(xgrid, kde(xgrid), alpha=0.35, color="steelblue")
ax.plot(xgrid, kde(xgrid), color="steelblue", lw=1.5)

ax.axvline(TIER_MILD,     color="#ffcc80", lw=1.5, ls="--", label=f"Mild threshold ({TIER_MILD} DH)")
ax.axvline(TIER_MODERATE, color="#ef9a9a", lw=1.5, ls="--", label=f"Moderate threshold ({TIER_MODERATE} DH)")
ax.axvline(TIER_SEVERE,   color="#b71c1c", lw=1.5, ls="--", label=f"Severe threshold ({TIER_SEVERE} DH)")

ax.set_xlabel("Annual Frost Degree-Hours")
ax.set_ylabel("Density")
ax.set_title(f"ERA5 Frost DH Distribution (1940–2024, n={len(df)})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Summary

| Metric | Value |
|--------|-------|
| Historical period (frost data) | 1940–2024 |
| Overlapping panel (with production) | 1961–2024 |
| Mild frost P(DH ≥ 50) | see Section 2 |
| Total Annual Expected Loss | see Section 6 |

**Key caveats:**
1. Production YoY change captures multi-factor weather noise — drought, hail, pollination failures all contribute. Frost is one driver.
2. The OLS relationship in Section 4 gives a directional signal; significance depends on panel length.
3. DH thresholds (50/150/300) are illustrative starting points — they should be calibrated against the payout bands in `trigger_params.yaml` for the final contract.